# 🧪 Lab: Hypothesis Testing

Aplicamos el procedimiento completo de prueba de hipótesis a dos datasets:
- **Challenge 1:** Datos Pokémon
- **Challenge 2:** Vivienda en California

In [1]:
import pandas as pd
import scipy.stats as st
from scipy.stats import mannwhitneyu, shapiro, kstest, levene

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv')
df.head()

,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,False
1,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,False
2,Venusaur,Grass,Poison,80,82,83,100,100,80,1,False
3,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,1,False
4,Charmander,Fire,NaN,39,52,43,60,50,65,1,False


---
## 🎮 Challenge 1 — Pokémon Data

---
### Challenge 1.1
### Queremos comprobar si los Pokémon tipo Dragon tienen, de media, más HP que los tipo Grass.

#### 1. Formulación de hipótesis

- H0: media HP Dragon = media HP Grass
- H1: media HP Dragon > media HP Grass

In [3]:
# 2. Extracción de muestras
dragon = df[df['Type 1'] == 'Dragon']['HP']
grass  = df[df['Type 1'] == 'Grass']['HP']

print(f'Dragon — n={len(dragon)}, media HP = {dragon.mean():.2f}')
print(f'Grass  — n={len(grass)},  media HP = {grass.mean():.2f}')

Dragon — n=32, media HP = 83.31
Grass  — n=70,  media HP = 67.27


In [4]:
# 3. Nivel de significancia
alpha = 0.05

#### 4. Comprobación de condiciones

In [5]:
# Comprobación de normalidad
# n Dragon < 5000 y n Grass < 5000 -> usamos Shapiro-Wilk
_, p_dragon = shapiro(dragon)
_, p_grass  = shapiro(grass)

print(f'Shapiro Dragon: p = {p_dragon:.4f}')
print(f'Shapiro Grass:  p = {p_grass:.4f}')

if p_dragon > alpha and p_grass > alpha:
    print('✅ Ambas muestras son normales -> comprobamos homogeneidad de varianza')
else:
    print('❌ Al menos una muestra no es normal -> usaremos test no paramétrico')

Shapiro Dragon: p = 0.3420
Shapiro Grass:  p = 0.1121
✅ Ambas muestras son normales -> comprobamos homogeneidad de varianza


In [6]:
# Comprobación homogeneidad de varianza (Levene, más robusto)
_, p_levene = levene(dragon, grass)
print(f'Levene p = {p_levene:.4f}')

if p_levene > alpha:
    print('✅ Varianzas homogéneas -> test paramétrico (equal_var=True)')
else:
    print('❌ Varianzas no homogéneas -> test paramétrico con equal_var=False')

Levene p = 0.1748
✅ Varianzas homogéneas -> test paramétrico (equal_var=True)


#### 5. Selección del estadístico

- Si normalidad ✅ → `st.ttest_ind(dragon, grass, equal_var=..., alternative='greater')`
- Si normalidad ❌ → `mannwhitneyu(dragon, grass, alternative='greater')`

In [7]:
# Aplicamos el test según resultado de comprobaciones
if p_dragon > alpha and p_grass > alpha:
    equal_var = p_levene > alpha
    resultado = st.ttest_ind(dragon, grass, equal_var=equal_var, alternative='greater')
    print('Test aplicado: Two Sample T-Test (paramétrico)')
else:
    resultado = mannwhitneyu(dragon, grass, alternative='greater')
    print('Test aplicado: Mann-Whitney U (no paramétrico)')

print(f'p-value: {resultado.pvalue:.4f}')

Test aplicado: Two Sample T-Test (paramétrico)
p-value: 0.0003


In [8]:
# 6. Toma de decisión
if resultado.pvalue < alpha:
    print('Se rechaza la hipótesis nula H0 -> Se acepta la hipótesis alternativa H1.')
else:
    print('Se acepta la hipótesis nula H0.')

Se rechaza la hipótesis nula H0 -> Se acepta la hipótesis alternativa H1.


#### 7. Interpretación de resultados

Los Pokémon tipo Dragon **SÍ / NO** tienen significativamente más HP que los tipo Grass, con un nivel de significancia del 5%.

---
### Challenge 1.2
### Queremos comprobar si los Pokémon Legendarios tienen diferentes stats (HP, Attack, Defense, Sp. Atk, Sp. Def, Speed) que los No-Legendarios.

#### 1. Formulación de hipótesis

- H0: media stat Legendarios = media stat No-Legendarios
- H1: media stat Legendarios != media stat No-Legendarios

In [9]:
# 2. Extracción de muestras
legendary     = df[df['Legendary'] == True]
non_legendary = df[df['Legendary'] == False]

print(f'Legendarios: {len(legendary)} | No-Legendarios: {len(non_legendary)}')

Legendarios: 65 | No-Legendarios: 735


In [10]:
# 3. Nivel de significancia
alpha = 0.05

#### 4. Comprobación de condiciones

n No-Legendarios > 50 → por el Teorema Central del Límite podemos asumir normalidad.

#### 5. Selección del estadístico

Dos grupos independientes, hipótesis bilateral (sin dirección) → `st.ttest_ind(..., alternative='two-sided')`

In [11]:
stats = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']

print(f'{"Stat":<12} {"Media Leg":>10} {"Media No-Leg":>14} {"p-value":>10} {"Decisión":>25}')
print('-' * 75)

for stat in stats:
    _, p_value = st.ttest_ind(legendary[stat], non_legendary[stat],
                              equal_var=False, alternative='two-sided')
    decision = 'Rechazamos H0' if p_value < alpha else 'Aceptamos H0'
    print(f'{stat:<12} {legendary[stat].mean():>10.2f} {non_legendary[stat].mean():>14.2f} {p_value:>10.4f} {decision:>25}')

Stat          Media Leg   Media No-Leg    p-value                  Decisión
---------------------------------------------------------------------------
HP                92.74          67.18     0.0000             Rechazamos H0
Attack           116.68          75.67     0.0000             Rechazamos H0
Defense           99.66          71.56     0.0000             Rechazamos H0
Sp. Atk          122.18          68.45     0.0000             Rechazamos H0
Sp. Def          105.94          68.89     0.0000             Rechazamos H0
Speed            100.18          65.46     0.0000             Rechazamos H0


#### 6 y 7. Toma de decisión e interpretación

Para todas las stats donde p-value < 0.05, rechazamos H0 → los Pokémon Legendarios tienen stats significativamente diferentes a los No-Legendarios, con un nivel de significancia del 5%.

---
## 🏠 Challenge 2 — California Housing

---
### Queremos comprobar si las casas cercanas a una escuela o hospital son más caras.

- Coordenadas escuela: (-118, 34)
- Coordenadas hospital: (-122, 37)
- Umbral de cercanía: distancia euclidiana < 0.50

#### 1. Formulación de hipótesis

- H0: precio medio casas cercanas = precio medio casas lejanas
- H1: precio medio casas cercanas > precio medio casas lejanas

In [12]:
import numpy as np

df2 = pd.read_csv('https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv')
df2.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


In [13]:
# 2. Extracción de muestras
# Calculamos distancia euclidiana a escuela y hospital
def euclidean_distance(lon, lat, ref_lon, ref_lat):
    return np.sqrt((lon - ref_lon)**2 + (lat - ref_lat)**2)

df2['dist_school']   = euclidean_distance(df2['longitude'], df2['latitude'], -118, 34)
df2['dist_hospital'] = euclidean_distance(df2['longitude'], df2['latitude'], -122, 37)

df2['near_facility'] = (df2['dist_school'] < 0.5) | (df2['dist_hospital'] < 0.5)

cerca = df2[df2['near_facility'] == True]['median_house_value']
lejos = df2[df2['near_facility'] == False]['median_house_value']

print(f'Casas cerca : {len(cerca):,} | media = ${cerca.mean():,.2f}')
print(f'Casas lejos : {len(lejos):,} | media = ${lejos.mean():,.2f}')

Casas cerca : 6,829 | media = $246,951.98
Casas lejos : 10,171 | media = $180,678.44


In [14]:
# 3. Nivel de significancia
alpha = 0.05

#### 4. Comprobación de condiciones

n > 5000 en ambos grupos → usamos Kolmogorov-Smirnov para comprobar normalidad.

In [15]:
# Comprobación de normalidad con KS (n > 5000)
_, p_cerca = kstest((cerca - cerca.mean()) / cerca.std(), 'norm')
_, p_lejos = kstest((lejos - lejos.mean()) / lejos.std(), 'norm')

print(f'KS cerca: p = {p_cerca:.4f}')
print(f'KS lejos: p = {p_lejos:.4f}')

if p_cerca > alpha and p_lejos > alpha:
    print('✅ Normales -> comprobamos homogeneidad de varianza')
else:
    print('❌ No normales -> usaremos test no paramétrico (Mann-Whitney U)')

KS cerca: p = 0.0000
KS lejos: p = 0.0000
❌ No normales -> usaremos test no paramétrico (Mann-Whitney U)


#### 5. Selección del estadístico

- Si normales ✅ → `st.ttest_ind(cerca, lejos, equal_var=..., alternative='greater')`
- Si no normales ❌ → `mannwhitneyu(cerca, lejos, alternative='greater')`

In [16]:
if p_cerca > alpha and p_lejos > alpha:
    _, p_lev = levene(cerca, lejos)
    equal_var = p_lev > alpha
    resultado = st.ttest_ind(cerca, lejos, equal_var=equal_var, alternative='greater')
    print('Test aplicado: Two Sample T-Test (paramétrico)')
else:
    resultado = mannwhitneyu(cerca, lejos, alternative='greater')
    print('Test aplicado: Mann-Whitney U (no paramétrico)')

print(f'p-value: {resultado.pvalue:.4f}')

Test aplicado: Mann-Whitney U (no paramétrico)
p-value: 0.0000


In [17]:
# 6. Toma de decisión
if resultado.pvalue < alpha:
    print('Se rechaza la hipótesis nula H0 -> Se acepta la hipótesis alternativa H1.')
else:
    print('Se acepta la hipótesis nula H0.')

Se rechaza la hipótesis nula H0 -> Se acepta la hipótesis alternativa H1.


#### 7. Interpretación de resultados

Las casas cercanas a una escuela o hospital **SÍ / NO** son significativamente más caras que las lejanas, con un nivel de significancia del 5%.